# B-07: spike-classifier diagnostics, recency features, early-warning threshold analysis (D-44)

B-06 (two-stage hurdle, D-43) was a documented negative result for the daily forecaster: the spike classifier
has high **recall** (0.58-0.93) but low **precision** (0.25-0.42 daily), so false-positive "spike" predictions
blend the noisy spike-only regressor into non-spike test points, inflating non-spike RMSE more than spike RMSE
shrinks. This notebook does three things, in sequence:

1. **Diagnose** the classifier's false positives/negatives at representative horizon/tower combos -- what do
   they look like (season, rainfall, grazing recency)?
2. **Add leak-free recency/clustering features** (days-since-last-spike, trailing spike count, trailing rolling
   max) computed from the gap-filled CH4 series using the same frozen q90 thresholds as B-06, and retest the
   full B-06 harness (classifier + 2 magnitude regressors + soft blend, plus plain RF/XGB) with them added --
   built and tested regardless of what the diagnostic finds (one bounded empirical test, no new HPO, D-41 norm).
3. **Early-warning threshold analysis**: a precision-recall curve across thresholds for the daily classifier,
   reframing its already-good recall as a standalone "elevated-emission-risk" decision-support signal,
   independent of whether it moves the regression R2.

Same data, partial pooling (T2+T4+T9 + dummies, D-30), CV, and B-03's already-tuned RF/XGB hyperparameters
throughout. All B-06 helper functions (`aligned`, `fit_reg`, `fit_clf`, `predict_hurdle`, `emit`, `run_horizon`)
are reused verbatim.

In [1]:
from pathlib import Path
import sys, datetime, numpy as np, pandas as pd
sys.path.insert(0, "../../src")
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              precision_score, recall_score, f1_score, precision_recall_curve)
from xgboost import XGBRegressor, XGBClassifier
import matplotlib.pyplot as plt
from evaluation.metrics import full_metrics
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
ALGOS=["RF","XGB"]
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]
DUM=["is_t2","is_t4","is_t9"]

matv2=pd.read_csv(HOURLY/"forecast_features_v2.csv",low_memory=False)
matv2["Datetime"]=pd.to_datetime(matv2["Datetime"],format="mixed")
AR_A=[c for c in matv2.columns if c.startswith("ar_")]
FX_A=[c for c in matv2.columns if c.startswith("fx")]

dv=pd.read_csv(HOURLY/"forecast_daily_v2.csv",low_memory=False)
dv["Datetime"]=pd.to_datetime(dv["Datetime"],format="mixed")
AR_B=[c for c in dv.columns if c.startswith("ar_")]
FX_B=[c for c in dv.columns if c.startswith("fx")]
print("hourly v2 rows",len(matv2),"| AR_A",len(AR_A),"| FX_A",len(FX_A))
print("daily  v2 rows",len(dv),"| AR_B",len(AR_B),"| FX_B",len(FX_B))

hourly v2 rows 210459 | AR_A 20 | FX_A 26
daily  v2 rows 8772 | AR_B 7 | FX_B 34


## 1  B-06 helpers, reused verbatim (spike thresholds + classifier/regressor harness)

In [2]:
def spike_thresholds(df, towers=(2,4,9)):
    th={}
    for t in towers:
        s=df[df.tower==t]
        tr=s[(s.Datetime.dt.year>=2018)&(s.Datetime.dt.year<=2021)]["y_observed"].dropna()
        th[t]=float(tr.quantile(0.90)) if len(tr)>=20 else np.nan
    return th

THRESH_A=spike_thresholds(matv2)
THRESH_B=spike_thresholds(dv)
print("Track A (hourly) q90 thresholds:",THRESH_A)
print("Track B (daily)  q90 thresholds:",THRESH_B)

def fit_reg(algo, tr, feat, track="A"):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestRegressor(n_estimators=500,n_jobs=-1,random_state=42,**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        m=XGBRegressor(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,**p)
    m.fit(Xi,tr["target"].values); return m,imp

def fit_clf(algo, tr, feat, track="A"):
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    y=tr["spike"].values.astype(int)
    if y.sum()<5 or y.sum()>len(y)-5:
        return None,imp
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestClassifier(n_estimators=500,n_jobs=-1,random_state=42,class_weight="balanced",**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        spw=float((y==0).sum())/max(1,(y==1).sum())
        m=XGBClassifier(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,
                         scale_pos_weight=spw,eval_metric="logloss",**p)
    m.fit(Xi,y); return m,imp

def predict_hurdle(clf,clf_imp,reg_spike,reg_spike_imp,reg_base,reg_base_imp,feat,X):
    p_spike = clf.predict_proba(clf_imp.transform(X[feat].values))[:,1] if clf is not None else np.zeros(len(X))
    yhat_spike = reg_spike.predict(reg_spike_imp.transform(X[feat].values))
    yhat_base  = reg_base.predict(reg_base_imp.transform(X[feat].values))
    return p_spike*yhat_spike + (1-p_spike)*yhat_base, p_spike

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def climatology(tr, te, unit):
    t=te["tower"].iloc[0]; trt=tr[tr.tower==t]
    if len(trt)<24: trt=tr
    gl=float(trt["target"].mean())
    if unit=="h":
        g=trt.assign(mo=trt.ttime.dt.month,hr=trt.ttime.dt.hour).groupby(["mo","hr"])["target"].mean()
        keys=list(zip(te.ttime.dt.month,te.ttime.dt.hour)); return np.array([g.get(k,gl) for k in keys])
    g=trt.assign(mo=trt.ttime.dt.month).groupby("mo")["target"].mean()
    return np.array([g.get(m,gl) for m in te.ttime.dt.month])

def aligned(T, h, unit, ar_cols, fx_cols, thresh):
    parts=[]; off=pd.Timedelta(hours=h) if unit=="h" else pd.Timedelta(days=h)
    for t,df in T.items():
        f=df[ar_cols+DUM].copy()
        for c in fx_cols: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["persist"]=df["y_gapfilled"]
        f["tower"]=t; f["ttime"]=df.index+off
        f["spike"]=(f["target"]>thresh.get(t,np.inf)).astype(float)
        f.loc[f["target"].isna(),"spike"]=np.nan
        parts.append(f)
    return pd.concat(parts)

def build_hourly_tables():
    T={t: matv2[matv2.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_A, FX_A

def build_daily_tables():
    T={t: dv[dv.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_B, FX_B

def emit(track,h,t,bydict,rp,rc,y_naive,event=None):
    rows=[]
    for model,(y,p) in bydict.items():
        r,mae,r2,mbe=metrics(y,p)
        fm=full_metrics(y,p,y_naive=y_naive)
        row=dict(track=track,horizon=h,tower=t,model=model,RMSE=round(r,3),MAE=round(mae,3),
            R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),n_test=len(y),
            skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            skill_clim=round(1-r/rc,3) if rc>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"],
            precision=np.nan,recall=np.nan,f1=np.nan,
            rmse_spike=np.nan,rmse_nonspike=np.nan,n_spike_test=np.nan)
        if event is not None and model.startswith("Hurdle"):
            row.update(event)
        rows.append(row)
    return rows

def run_horizon(track, h, unit, ar_cols, fx_cols, T, thresh):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols,thresh); rows=[]
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    tr_base=tr[tr.spike==0]; tr_spike=tr[tr.spike==1]
    fitted_reg={a:fit_reg(a,tr,feat,track) for a in ALGOS}
    fitted_base={a:fit_reg(a,tr_base,feat,track) for a in ALGOS}
    fitted_spike={a:fit_reg(a,tr_spike,feat,track) for a in ALGOS}
    fitted_clf={a:fit_clf(a,tr,feat,track) for a in ALGOS}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        bd={"persist":(te["target"].values,te["persist"].values),
            "clim":(te["target"].values,climatology(tr,te,unit))}
        for a in ALGOS:
            m,imp=fitted_reg[a]; bd[a]=(te["target"].values,m.predict(imp.transform(te[feat].values)))
        rp=rmse(*bd["persist"]); rc=rmse(*bd["clim"]); y_naive=bd["persist"][1]
        for a in ALGOS:
            clf,clf_imp=fitted_clf[a]; rb,rb_imp=fitted_base[a]; rs,rs_imp=fitted_spike[a]
            yhat,p_spike=predict_hurdle(clf,clf_imp,rs,rs_imp,rb,rb_imp,feat,te)
            bd[f"Hurdle-{a}"]=(te["target"].values,yhat)
            y_true_cls=te["spike"].values
            event=dict(n_spike_test=int(np.nansum(y_true_cls)))
            if clf is not None and np.nansum(y_true_cls)>0 and np.nansum(1-y_true_cls)>0:
                pred_cls=(p_spike>0.5).astype(int)
                event["precision"]=round(float(precision_score(y_true_cls,pred_cls,zero_division=0)),3)
                event["recall"]=round(float(recall_score(y_true_cls,pred_cls,zero_division=0)),3)
                event["f1"]=round(float(f1_score(y_true_cls,pred_cls,zero_division=0)),3)
            spk_mask=y_true_cls==1; nsp_mask=y_true_cls==0
            if spk_mask.sum()>0: event["rmse_spike"]=round(rmse(te["target"].values[spk_mask],yhat[spk_mask]),3)
            if nsp_mask.sum()>0: event["rmse_nonspike"]=round(rmse(te["target"].values[nsp_mask],yhat[nsp_mask]),3)
            rows+=emit(track,h,t,{f"Hurdle-{a}":bd[f"Hurdle-{a}"]},rp,rc,y_naive,event=event)
        rows+=emit(track,h,t,{k:v for k,v in bd.items() if not k.startswith("Hurdle")},rp,rc,y_naive)
    acc={k:([],[]) for k in ["persist","clim"]+ALGOS+[f"Hurdle-{a}" for a in ALGOS]}
    for cut,s,e in T2_FOLDS:
        trf=A[(A.ttime<=pd.Timestamp(cut)) & A.target.notna()]
        trf_base=trf[trf.spike==0]; trf_spike=trf[trf.spike==1]
        te=A[(A.tower==2)&(A.ttime>=pd.Timestamp(s))&(A.ttime<=pd.Timestamp(e))&A.target.notna()]
        if len(te)<5 or len(trf)<50: continue
        y=te["target"].values
        acc["persist"][0].append(y); acc["persist"][1].append(te["persist"].values)
        acc["clim"][0].append(y); acc["clim"][1].append(climatology(trf,te,unit))
        for a in ALGOS:
            m,imp=fit_reg(a,trf,feat,track); acc[a][0].append(y); acc[a][1].append(m.predict(imp.transform(te[feat].values)))
            clf,clf_imp=fit_clf(a,trf,feat,track)
            rb,rb_imp=fit_reg(a,trf_base,feat,track) if len(trf_base)>=10 else fit_reg(a,trf,feat,track)
            rs,rs_imp=fit_reg(a,trf_spike,feat,track) if len(trf_spike)>=10 else fit_reg(a,trf,feat,track)
            yhat,_=predict_hurdle(clf,clf_imp,rs,rs_imp,rb,rb_imp,feat,te)
            acc[f"Hurdle-{a}"][0].append(y); acc[f"Hurdle-{a}"][1].append(yhat)
    if acc["persist"][0]:
        bd={k:(np.concatenate(v[0]),np.concatenate(v[1])) for k,v in acc.items()}
        rp=rmse(*bd["persist"]); rc=rmse(*bd["clim"])
        rows+=emit(track,h,2,bd,rp,rc,bd["persist"][1])
    return rows

TA,AR_A_,FX_A_=build_hourly_tables(); TB,AR_B_,FX_B_=build_daily_tables()
print("ready")

Track A (hourly) q90 thresholds: {2: 71.457395214, 4: 78.99610909200005, 9: 109.72678194800001}
Track B (daily)  q90 thresholds: {2: 81.90970496835237, 4: 71.08691920402013, 9: 116.4771911111111}
ready


## 2  Step 1 -- diagnose B-06 classifier false positives/negatives

Representative combos only (bounded, per scope guard): daily h=1, h=14; hourly h=1, h=24; Towers 4/9; both
algos. For each test row, capture the true spike label, predicted probability/class, and context (month,
forecast precip, days-since-grazing, growing-season flag) to see whether false positives cluster around
recent rain or grazing, or are seasonal.

In [3]:
def diagnose_rows(track, h, unit, ar_cols, fx_cols, T, thresh, precip_col, grazing_col="fx_days_since_grazing", grow_col="fx_is_growing"):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols,thresh)
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    rows=[]
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()].copy()
        if len(te)<10: continue
        for algo in ALGOS:
            clf,clf_imp=fit_clf(algo,tr,feat,track)
            if clf is None: continue
            p_spike=clf.predict_proba(clf_imp.transform(te[feat].values))[:,1]
            pred=(p_spike>0.5).astype(int); true=te["spike"].values.astype(int)
            # default="" avoids NumPy 2.x dtype mismatch between str choices and int default
            outcome=np.select([(true==1)&(pred==1),(true==0)&(pred==1),(true==1)&(pred==0),(true==0)&(pred==0)],
                               ["TP","FP","FN","TN"], default="")
            d=pd.DataFrame({
                "track":track,"horizon":h,"tower":t,"algo":algo,"ttime":te["ttime"].values,
                "p_spike":p_spike,"true":true,"pred":pred,"outcome":outcome,
                "month":te["ttime"].dt.month.values,
                "precip":te[precip_col].values if precip_col in te.columns else np.nan,
                "days_since_grazing":te[grazing_col].values if grazing_col in te.columns else np.nan,
                "is_growing":te[grow_col].values if grow_col in te.columns else np.nan})
            rows.append(d)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

diag_B=pd.concat([diagnose_rows("B",h,"D",AR_B_,FX_B_,TB,THRESH_B,"fx_PRECIP_sum") for h in [1,14]],ignore_index=True)
diag_A=pd.concat([diagnose_rows("A",h,"h",AR_A_,FX_A_,TA,THRESH_A,"fx_Precipitation (mm)") for h in [1,24]],ignore_index=True)
DIAG=pd.concat([diag_B,diag_A],ignore_index=True)
print("diagnostic rows:",len(DIAG))

print("=== outcome counts by track/tower/algo/horizon ===")
print(DIAG.groupby(["track","tower","algo","horizon","outcome"]).size().unstack(fill_value=0).to_string())

print("\n=== mean context by outcome (track B, daily) ===")
print(DIAG[DIAG.track=="B"].groupby("outcome")[["precip","days_since_grazing","is_growing"]].mean().round(2).to_string())

print("\n=== mean context by outcome (track A, hourly) ===")
print(DIAG[DIAG.track=="A"].groupby("outcome")[["precip","days_since_grazing","is_growing"]].mean().round(2).to_string())

print("\n=== FP rate by month (track B, daily) ===")
sub=DIAG[DIAG.track=="B"]
fp_rate=sub.assign(is_fp=(sub.outcome=="FP").astype(int)).groupby("month")["is_fp"].mean().round(3)
print(fp_rate.to_string())

diagnostic rows: 55928
=== outcome counts by track/tower/algo/horizon ===
outcome                    FN   FP    TN   TP
track tower algo horizon                     
A     4     RF   1        603  116  6259  131
                 24       687   64  6311   47
            XGB  1        512  340  6035  222
                 24       572  283  6092  162
      9     RF   1        410  183  5109  146
                 24       481  117  5175   75
            XGB  1        333  394  4898  223
                 24       368  386  4906  188
B     4     RF   1         15   72   428   52
                 14        16   82   418   51
            XGB  1          8   94   406   59
                 14         5  120   380   62
      9     RF   1          7   72   343   36
                 14        18   71   344   25
            XGB  1          3   92   323   40
                 14         5  117   298   38

=== mean context by outcome (track B, daily) ===
         precip  days_since_grazing  is_growing


## 3  Step 2 -- leak-free recency/clustering features

Computed from the **gap-filled continuous CH4 series** at each tower (same source as the existing
`ar_ch4_*lag*`/`ar_ch4_*rm*` features), using the same frozen per-tower q90 threshold as B-06. Strictly causal:
each feature at origin row `i` only uses `y_gapfilled[0..i]` -- never the horizon-shifted target, which is
applied afterwards by `aligned()`. Treated as `ar_` features (origin-time, not shifted by `-h`), directly
parallel to the existing AR design.

- `ar_days_since_spike` -- steps since the gap-filled series last exceeded threshold, as of origin (capped at
  9999 for origin rows with no prior spike in history).
- `ar_spike_count_<w>` -- trailing count of threshold-exceeding points in window `w` (inclusive of origin).
- `ar_rolling_max_<w>` -- trailing rolling max of the gap-filled series in window `w` (complements the existing
  `ar_ch4_rmean24`/`drm7` rolling *mean* with a recent-peak signal).

Windows: hourly w in {24, 168} (1 day, 7 days); daily w in {7, 28} (1 week, 4 weeks).

In [4]:
def add_recency_features(T, thresh, w1, w2):
    T2={}
    for t,df in T.items():
        df=df.copy()
        s=df["y_gapfilled"]
        is_spike=(s>thresh[t]).astype(int)
        sv=is_spike.values; n=len(sv)
        last_pos=np.empty(n,dtype=np.int64); pos=-1
        for i in range(n):
            if sv[i]==1: pos=i
            last_pos[i]=pos
        days_since=np.where(last_pos<0,9999,np.arange(n)-last_pos)
        df["ar_days_since_spike"]=days_since
        df[f"ar_spike_count_{w1}"]=is_spike.rolling(w1,min_periods=1).sum().values
        df[f"ar_spike_count_{w2}"]=is_spike.rolling(w2,min_periods=1).sum().values
        df[f"ar_rolling_max_{w1}"]=s.rolling(w1,min_periods=1).max().values
        df[f"ar_rolling_max_{w2}"]=s.rolling(w2,min_periods=1).max().values
        T2[t]=df
    return T2

TA_R=add_recency_features(TA, THRESH_A, 24, 168)
TB_R=add_recency_features(TB, THRESH_B, 7, 28)
RECENCY_A=["ar_days_since_spike","ar_spike_count_24","ar_spike_count_168","ar_rolling_max_24","ar_rolling_max_168"]
RECENCY_B=["ar_days_since_spike","ar_spike_count_7","ar_spike_count_28","ar_rolling_max_7","ar_rolling_max_28"]
AR_A_R=AR_A_+RECENCY_A; AR_B_R=AR_B_+RECENCY_B

# leak-free spot-check: days-since-spike must never decrease except resetting to 0 exactly at a spike row
chk=TB_R[4]
bad=((chk["ar_days_since_spike"].diff()!=1) & (chk["ar_days_since_spike"]!=0)).sum()
print("Tower 4 daily ar_days_since_spike non-monotonic-or-reset rows (expect 0):", bad)
print(chk[["y_gapfilled","ar_days_since_spike","ar_spike_count_7","ar_rolling_max_7"]].head(15).to_string())

Tower 4 daily ar_days_since_spike non-monotonic-or-reset rows (expect 0): 4
            y_gapfilled  ar_days_since_spike  ar_spike_count_7  ar_rolling_max_7
Datetime                                                                        
2017-01-01    14.863479                 9999               0.0         14.863479
2017-01-02    21.314554                 9999               0.0         21.314554
2017-01-03    20.989712                 9999               0.0         21.314554
2017-01-04    15.136346                 9999               0.0         21.314554
2017-01-05   363.380304                    0               1.0        363.380304
2017-01-06   179.990250                    0               2.0        363.380304
2017-01-07   150.133854                    0               3.0        363.380304
2017-01-08   351.137275                    0               4.0        363.380304
2017-01-09    81.819071                    0               5.0        363.380304
2017-01-10     7.754146          

## 4  Step 3 -- retest with recency features added (no new HPO, single isolated lever)

Same harness (`run_horizon`), same hyperparameters, only the feature set changes (`AR_A_R`/`AR_B_R` in place of
`AR_A_`/`AR_B_`). New rows tagged `RF-Recency`/`XGB-Recency`/`Hurdle-RF-Recency`/`Hurdle-XGB-Recency` -- directly
comparable to B-06's and B-03's existing numbers.

In [5]:
ALL_R=[]
for h in HORIZONS["A"]:
    ALL_R+=run_horizon("A",h,"h",AR_A_R,FX_A_,TA_R,THRESH_A); print("Track A h=",h,"done (recency)",flush=True)
for h in HORIZONS["B"]:
    ALL_R+=run_horizon("B",h,"D",AR_B_R,FX_B_,TB_R,THRESH_B); print("Track B d=",h,"done (recency)",flush=True)
R_RECENCY=pd.DataFrame(ALL_R)
R_RECENCY=R_RECENCY[R_RECENCY.model.isin(["RF","XGB","Hurdle-RF","Hurdle-XGB"])].copy()
R_RECENCY["model"]=R_RECENCY["model"]+"-Recency"
print("recency rows",len(R_RECENCY))

Track A h= 1 done (recency)


Track A h= 6 done (recency)


Track A h= 12 done (recency)


Track A h= 24 done (recency)


Track A h= 48 done (recency)


Track B d= 1 done (recency)


Track B d= 3 done (recency)


Track B d= 7 done (recency)


Track B d= 14 done (recency)


recency rows 108


In [6]:
b06=pd.read_csv(RESULTS/"b06_summary.csv")
b03=pd.read_csv(RESULTS/"b03_summary.csv")
cmp_keys=["track","horizon","tower"]

def side_by_side(model_base):
    a=b03[(b03.model==model_base)&b03.tower.isin([4,9])][cmp_keys+["R2"]].rename(columns={"R2":"R2_b03"})
    b=b06[(b06.model==model_base)&b06.tower.isin([4,9])][cmp_keys+["R2"]].rename(columns={"R2":"R2_b06"})
    c=R_RECENCY[(R_RECENCY.model==model_base+"-Recency")&R_RECENCY.tower.isin([4,9])][cmp_keys+["R2"]].rename(columns={"R2":"R2_b07_recency"})
    m=a.merge(b,on=cmp_keys,how="outer").merge(c,on=cmp_keys,how="outer")
    return m.sort_values(cmp_keys)

print("=== RF: B-03 (no spike-aware) vs B-06 (hurdle) vs B-07 (RF + recency, plain) ===")
print(side_by_side("RF").to_string(index=False))
print("\n=== XGB ===")
print(side_by_side("XGB").to_string(index=False))
print("\n=== Hurdle-RF (B-06 vs B-07 hurdle+recency) ===")
print(side_by_side("Hurdle-RF").to_string(index=False))
print("\n=== Hurdle-XGB ===")
print(side_by_side("Hurdle-XGB").to_string(index=False))

print("\n=== Recency classifier precision/recall/f1 (Hurdle-*-Recency, towers 4/9) ===")
evr=R_RECENCY[(R_RECENCY.tower.isin([4,9]))&(R_RECENCY.model.str.startswith("Hurdle"))]
print(evr.pivot_table(index=["track","tower","model"],columns="horizon",values=["precision","f1"]).round(3).to_string())

=== RF: B-03 (no spike-aware) vs B-06 (hurdle) vs B-07 (RF + recency, plain) ===
track  horizon  tower  R2_b03  R2_b06  R2_b07_recency
    A        1      4   0.136   0.136           0.162
    A        1      9   0.159   0.159           0.145
    A        6      4   0.044   0.044           0.051
    A        6      9   0.084   0.084           0.083
    A       12      4   0.078   0.078           0.079
    A       12      9   0.081   0.081           0.080
    A       24      4   0.049   0.049           0.054
    A       24      9   0.087   0.087           0.088
    A       48      4   0.032   0.032           0.042
    A       48      9   0.055   0.055           0.050
    B        1      4   0.357   0.357           0.359
    B        1      9   0.388   0.388           0.375
    B        3      4   0.345   0.345           0.343
    B        3      9   0.350   0.350           0.343
    B        7      4   0.287   0.287           0.258
    B        7      9   0.364   0.364           0.341
 

## 5  Step 4 -- early-warning threshold analysis (daily track)

Precision-recall curve across thresholds for the daily-track recency-augmented classifier (h=1 and h=14,
Towers 4/9, both algos), independent of the regression blend. Recommended operating point: the most
conservative threshold that still achieves recall >= 0.8 (deliberately favouring recall -- false alarms are
cheaper than missed elevated-emission days in a farm-management early-warning use case).

In [7]:
def classifier_probs(track, h, unit, ar_cols, fx_cols, T, thresh, algo):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols,thresh)
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    out={}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        clf,clf_imp=fit_clf(algo,tr,feat,track)
        if clf is None: continue
        p=clf.predict_proba(clf_imp.transform(te[feat].values))[:,1]
        out[t]=(te["spike"].values.astype(int), p)
    return out

PR={}
for h in [1,14]:
    for algo in ALGOS:
        PR[(h,algo)]=classifier_probs("B",h,"D",AR_B_R,FX_B_,TB_R,THRESH_B,algo)

rows=[]
fig,axes=plt.subplots(2,2,figsize=(10,8),sharex=True,sharey=True)
for i,h in enumerate([1,14]):
    for j,algo in enumerate(ALGOS):
        ax=axes[i,j]
        for t,col in zip([4,9],["C0","C1"]):
            if t not in PR[(h,algo)]: continue
            y,p=PR[(h,algo)][t]
            prec,rec,thr=precision_recall_curve(y,p)
            ax.plot(rec,prec,label=f"Tower {t}",color=col)
            # recommended: most conservative threshold with recall>=0.8
            ok=np.where(rec[:-1]>=0.8)[0]
            if len(ok):
                k=ok[np.argmax(thr[ok])]
                ax.scatter([rec[k]],[prec[k]],color=col,marker="x",s=80)
                rows.append(dict(horizon=h,algo=algo,tower=t,threshold=round(float(thr[k]),3),
                                  recall=round(float(rec[k]),3),precision=round(float(prec[k]),3)))
            # default 0.5 operating point
            k2=int(np.argmin(np.abs(thr-0.5))) if len(thr) else None
            if k2 is not None:
                ax.scatter([rec[k2]],[prec[k2]],color=col,marker="o",s=40,facecolors="none")
        ax.set_title(f"h={h}d, {algo}"); ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(fontsize=8)
fig.suptitle("Daily-track PR curves (recency classifier) -- x = recommended (recall>=0.8), o = default (p=0.5)")
fig.tight_layout()
fig.savefig(RESULTS/"figures"/"b07_pr_curve.png",dpi=120)
plt.close(fig)

OPS=pd.DataFrame(rows)
print("=== Recommended early-warning operating points (recall>=0.8 target) ===")
print(OPS.to_string(index=False))
print("\nsaved results/figures/b07_pr_curve.png")

=== Recommended early-warning operating points (recall>=0.8 target) ===
 horizon algo  tower  threshold  recall  precision
       1   RF      4      0.459   0.806      0.425
       1   RF      9      0.513   0.814      0.357
       1  XGB      4      0.675   0.806      0.432
       1  XGB      9      0.656   0.814      0.324
      14   RF      4      0.498   0.806      0.391
      14   RF      9      0.397   0.814      0.280
      14  XGB      4      0.686   0.806      0.391
      14  XGB      9      0.613   0.814      0.302

saved results/figures/b07_pr_curve.png


## 6  Save b07_summary.csv and append B07 rows to benchmarks.csv

In [8]:
SUMMARY=pd.concat([R_RECENCY, OPS.assign(track="B",model="early_warning_op_point",n_test=np.nan)],ignore_index=True,sort=False)
SUMMARY.to_csv(RESULTS/"b07_summary.csv",index=False)
print("saved results/b07_summary.csv, rows:",len(SUMMARY))

bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B07"]
rows=[]
for _,r in R_RECENCY.iterrows():
    rows.append({"replication":"B07","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"enriched_v2 + leak-free recency/clustering features (days_since_spike, spike_count, rolling_max)",
        "track":r["track"],"horizon":int(r["horizon"]),
        "split":"fc_main" if r.tower in (4,9) else "fc_t2folds",
        "R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],"MBE":r["MBE"],"n_test":int(r["n_test"]),
        "skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "precision":r["precision"],"recall":r["recall"],"f1":r["f1"],
        "date":today,"notes":"B07 spike diagnostics + recency features added to B-06 hurdle/plain RF/XGB (no new HPO); compare vs B03/B06 (D-44); WAPE/MASE/sMAPE/MAPE added D-44b"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B07 rows. Total {len(comb)}.")

saved results/b07_summary.csv, rows: 116
Wrote 108 B07 rows. Total 3665.
